# GitHub statistics

Shows the number of newly created repositories and the number of commits per month for the authenticated user.

Set your token before starting Jupyter: `export GITHUB_TOKEN=<your token>` (or enter it when prompted).

In [ ]:
import os, time, getpass
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')

token = os.environ.get('GITHUB_TOKEN') or getpass.getpass('GitHub API token: ')
session = requests.Session()
session.headers.update({'Authorization': f'Bearer {token}', 'Accept': 'application/vnd.github+json'})
API = 'https://api.github.com'

def get(url, **params):
    while True:
        r = session.get(url, params=params)
        if r.status_code in (403, 429) and r.headers.get('X-RateLimit-Remaining') == '0' or r.status_code == 429:
            reset = int(r.headers.get('X-RateLimit-Reset', time.time() + 60))
            time.sleep(max(reset - time.time(), 1) + 1)
            continue
        r.raise_for_status()
        return r

user = get(f'{API}/user').json()['login']
print('User:', user)

def plot_monthly(series, title, ylabel):
    fig, ax = plt.subplots(figsize=(14, 4))
    sns.barplot(x=list(range(len(series))), y=series.values, color=sns.color_palette()[0], ax=ax)
    ticks = [i for i, d in enumerate(series.index) if d.month == 1]
    ax.set_xticks(ticks)
    ax.set_xticklabels([series.index[i].year for i in ticks])
    ax.set_title(title)
    ax.set_xlabel('')
    ax.set_ylabel(ylabel)
    sns.despine(ax=ax)
    plt.tight_layout()
    plt.show()

## Repositories created per month

In [ ]:
repos, page = [], 1
while True:
    batch = get(f'{API}/user/repos', affiliation='owner', per_page=100, page=page).json()
    if not batch:
        break
    repos += batch
    page += 1

created = pd.to_datetime([r['created_at'] for r in repos])
repos_per_month = pd.Series(1, index=created).resample('MS').sum()
repos_per_month = repos_per_month.reindex(
    pd.date_range(repos_per_month.index.min(), pd.Timestamp.now(tz='UTC').normalize().replace(day=1), freq='MS'), fill_value=0)

plot_monthly(repos_per_month, 'New repositories per month', 'repositories')
print('Total repositories:', len(repos))

## Commits per month

Uses the commit search API (default branches only), one query per month. The search API is limited to 30 requests per minute, so this may take a while.

In [ ]:
months = pd.date_range(repos_per_month.index.min(), pd.Timestamp.now(tz='UTC'), freq='MS')
counts = {}
for start in months:
    end = (start + pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d')
    q = f'author:{user} author-date:{start:%Y-%m-%d}..{end}'
    counts[start] = get(f'{API}/search/commits', q=q, per_page=1).json()['total_count']
    time.sleep(2.1)

commits_per_month = pd.Series(counts)
plot_monthly(commits_per_month, 'Commits per month', 'commits')
print('Total commits:', commits_per_month.sum())